# Anchor OS Qwen3-0.6B router training
Runs the profile-conditioned LoRA workflow from the `model-router` branch. Use a GPU runtime. The dataset contains synthetic, non-confidential prompts and placeholder model profiles.

In [ ]:
!nvidia-smi
!git clone --branch model-router --single-branch https://github.com/shahidhustles/anchor-os.git /content/anchor-os
%cd /content/anchor-os
!python -m pip install -q -r packages/model-router/training/requirements.txt
!python packages/model-router/training/generate_data.py

Upload `checkpoint-300.zip` when prompted. It contains only LoRA training state and uses synthetic training data.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
uploaded = files.upload()
if set(uploaded) != {'checkpoint-300.zip'}:
    raise ValueError('Upload exactly checkpoint-300.zip')
archive = Path('/content/checkpoint-300.zip')
archive.write_bytes(uploaded['checkpoint-300.zip'])
resume_root = Path('/content/router-resume')
shutil.unpack_archive(archive, resume_root)
checkpoint = next(resume_root.rglob('trainer_state.json')).parent
print('Resume checkpoint:', checkpoint)

In [ ]:
!python -u packages/model-router/training/run_pipeline.py --resume "{checkpoint}"

The pipeline stops before the test split if validation misses 90% policy agreement or 100% accepted outputs. Download the artifacts after the cell above finishes.

In [ ]:
import shutil
from google.colab import files
artifact = shutil.make_archive('/content/qwen3-router-artifacts', 'zip', '/content/anchor-os/packages/model-router/artifacts/qwen3-router')
files.download(artifact)